# EPU_Locations_vs_CTF

This notebook requires access to:
1. An EPU session (with connected atlas session) 
2. Micrograph parameters .csv file downloaded from CryoSPARC. 

To obtain (2), build a Curate Exposures job, and then click "Download table CSV" (at the top of the micrographs table, next to the Filter dropdown)

The outputs of this job are images with one dot per micrograph, arrayed on a 2D grid resembling the atlas and colored by CTF Fit

## Imports and User Parameters

The user must provide base folders for session_dir, atlas_root, csv_path, and out_dir.

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Optional, Tuple, List
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from pathlib import Path

In [ ]:
## FILL ME IN

# EPU session directory (contains Images-Disc1)
SESSION_DIR = "/path/to/session/dir"

# Corresponding atlas directory (contains Atlas, Sample0, etc.)
ATLAS_ROOT  = "/path/to/atlas/dir"

# CSV file with CryoSPARC CTFs
csv_path = "path/to/csv/file.csv"

# output
OUT_DIR = "/path/to/output/directory/for/saved/images"

# "atlas plane" size used in annotate_atlas.py for scaling overlays. Shouldn't have to change. 
ATLAS_FULL_RES_PX = 4005.0

## PART I : Parse micrograph locations from EPU on atlas-level grid

### Helpers from epu_generate_report

Run this cell, shouldn't have to change anything

In [ ]:
## Annotate_atlas
def localname(tag):
    return tag.split('}')[-1] if '}' in tag else tag

def direct_child_by_localname(elem, name):
    for ch in list(elem):
        if localname(ch.tag) == name:
            return ch
    return None

def direct_child_text(elem, name):
    ch = direct_child_by_localname(elem, name)
    return ch.text.strip() if (ch is not None and ch.text is not None) else None

def find_descendant_first(elem, name):
    for e in elem.iter():
        if localname(e.tag) == name:
            return e
    return None

def atlas_id_from_epu_dm(session_dir: str):
    dm_path = os.path.join(session_dir, "EpuSession.dm")
    if not os.path.isfile(dm_path):
        return None
    try:
        root = ET.parse(dm_path).getroot()
    except Exception:
        return None
    for elem in root.iter():
        if localname(elem.tag).lower() == "atlasid":
            txt = (elem.text or "").strip()
            return txt if txt else None
    return None

def sample_from_atlas_id(atlas_id_text: str):
    if not atlas_id_text:
        return None
    parts = [p for p in re.split(r"[\\/]+", atlas_id_text.strip().strip('"').strip("'")) if p]
    for p in parts:
        if re.match(r"(?i)^sample\d+$", p):
            return p
    return None

def parse_atlas_nodes_precise(atlas_dm_path):
    """
    Parse Atlas.dm and extract atlas node metadata.
    """
    tree = ET.parse(atlas_dm_path)
    root = tree.getroot()
    nodes, pairs = {}, []

    for container in root.iter():
        pos = find_descendant_first(container, 'PositionOnTheAtlas')
        if pos is None:
            continue

        center = direct_child_by_localname(pos, 'Center')
        if center is None:
            continue
        ax_text = direct_child_text(center, 'x')
        ay_text = direct_child_text(center, 'y')
        rot_text = direct_child_text(pos, 'Rotation')
        shear_text = direct_child_text(pos, 'Shear')
        if ax_text is None or ay_text is None:
            continue

        # Stage position (aligned or raw)
        stage = find_descendant_first(container, 'StagePosition')
        if stage is None:
            stage = find_descendant_first(container, 'AlignedStagePosition')
        if stage is None:
            continue
        sx_text = direct_child_text(stage, 'X')
        sy_text = direct_child_text(stage, 'Y')
        if sx_text is None or sy_text is None:
            continue

        # Node ID
        node_id = None
        key_el = find_descendant_first(container, 'key')
        if key_el is not None and key_el.text:
            try:
                node_id = int(key_el.text.strip())
            except ValueError:
                node_id = None
        if node_id is None:
            id_el = find_descendant_first(container, 'Id')
            if id_el is not None and id_el.text:
                try:
                    node_id = int(id_el.text.strip())
                except ValueError:
                    node_id = None

        # Convert basics
        try:
            sx = float(sx_text)
            sy = float(sy_text)
            ax = float(ax_text)
            ay = float(ay_text)
        except ValueError:
            continue

        rot = None
        if rot_text is not None:
            try:
                rot = float(rot_text)
            except ValueError:
                rot = None

        shear = None
        if shear_text is not None:
            try:
                shear = float(shear_text)
            except ValueError:
                shear = None

        # Read atlas-pixel Size
        size_el = find_descendant_first(pos, 'Size')
        sw = sh = None
        if size_el is not None:
            sw_text = direct_child_text(size_el, 'width')
            sh_text = direct_child_text(size_el, 'height')
            try:
                sw = float(sw_text) if sw_text is not None else None
                sh = float(sh_text) if sh_text is not None else None
            except ValueError:
                sw = sh = None

        # Physical size (meters)
        phys_el = find_descendant_first(pos, 'Physical')
        pw = ph = None
        if phys_el is not None:
            pw_text = direct_child_text(phys_el, 'width')
            ph_text = direct_child_text(phys_el, 'height')
            try:
                pw = float(pw_text) if pw_text is not None else None
                ph = float(ph_text) if ph_text is not None else None
            except ValueError:
                pw = ph = None

        pairs.append((sx, sy, ax, ay))
        if node_id is not None:
            nodes[node_id] = {
                'stage_x': sx, 'stage_y': sy,
                'atlas_x': ax, 'atlas_y': ay,
                'rotation': rot, 'shear': shear,
                'size_w_px': sw, 'size_h_px': sh,
                'phys_w_m': pw, 'phys_h_m': ph,
            }

    if not pairs:
        raise RuntimeError('No pairs parsed in Atlas.dm')
    return nodes, pairs

def fit_stage_to_atlas_affine(pairs):
    S = np.asarray([[sx, sy, 1.0] for sx, sy, _, _ in pairs], dtype=np.float64)
    A = np.asarray([[ax, ay] for _, _, ax, ay in pairs], dtype=np.float64)
    M, _, _, _ = np.linalg.lstsq(S, A, rcond=None)

    def mapper(sx, sy):
        return tuple((np.array([sx, sy, 1.0]) @ M).tolist())

    return mapper, M

def parse_gridsquare_xml(xml_path):
    """
    Namespace-agnostic GridSquare XML parsing for stage X,Y (meters).
    """
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return {'stage_x': None, 'stage_y': None}
    sx = sy = None
    stage_el = find_descendant_first(root, 'stage')
    if stage_el is not None:
        pos_el = find_descendant_first(stage_el, 'Position')
        if pos_el is not None:
            sx_txt = direct_child_text(pos_el, 'X') or direct_child_text(pos_el, '_x')
            sy_txt = direct_child_text(pos_el, 'Y') or direct_child_text(pos_el, '_y')
            if sx_txt and sy_txt:
                sx, sy = sx_txt, sy_txt
    if sx is None or sy is None:
        pos_any = find_descendant_first(root, 'Position')
        if pos_any is not None:
            sx_txt = direct_child_text(pos_any, 'X') or direct_child_text(pos_any, '_x')
            sy_txt = direct_child_text(pos_any, 'Y') or direct_child_text(pos_any, '_y')
            if sx_txt and sy_txt:
                sx, sy = sx_txt, sy_txt
    return {'stage_x': float(sx) if sx is not None else None,
            'stage_y': float(sy) if sy is not None else None}

def map_grids_to_atlas(atlas_root, screening_root, check_node_center=True, fill_rotation='median'):
    # Prefer the sample referenced by EpuSession.dm <AtlasId>
    preferred_sample = sample_from_atlas_id(atlas_id_from_epu_dm(screening_root))

    # Build candidate Atlas.dm paths in priority order
    candidates = []
    if preferred_sample:
        candidates.append((preferred_sample, os.path.join(atlas_root, preferred_sample, "Atlas", "Atlas.dm")))

    candidates.append(("Sample0", os.path.join(atlas_root, "Sample0", "Atlas", "Atlas.dm")))

    # try any other SampleN
    try:
        for s in sorted(os.listdir(atlas_root)):
            if not re.match(r"(?i)^sample\d+$", s):
                continue
            if s in (preferred_sample, "Sample0"):
                continue
            candidates.append((s, os.path.join(atlas_root, s, "Atlas", "Atlas.dm")))
    except Exception:
        pass

    # last resort: atlas_root/Atlas/Atlas.dm (some datasets have this)
    candidates.append(("Atlas", os.path.join(atlas_root, "Atlas", "Atlas.dm")))

    nodes = pairs = None
    used_sample = None
    used_dm_path = None
    last_err = None

    for sample_name, atlas_dm_path in candidates:
        if not os.path.isfile(atlas_dm_path):
            continue
        try:
            nodes, pairs = parse_atlas_nodes_precise(atlas_dm_path)
            used_sample = sample_name
            used_dm_path = atlas_dm_path
            break
        except Exception as e:
            last_err = e
            continue

    if nodes is None or pairs is None:
        raise RuntimeError(f"Could not parse any Atlas.dm under {atlas_root}. Last error: {last_err}")

    stage_to_atlas, M = fit_stage_to_atlas_affine(pairs)

    discs = sorted([d for d in glob.glob(os.path.join(screening_root, 'Images-Disc*')) if os.path.isdir(d)]) or [screening_root]
    rows = []
    for disc in discs:
        grid_dirs = sorted([d for d in glob.glob(os.path.join(disc, 'GridSquare*')) if os.path.isdir(d)])
        for gdir in grid_dirs:
            xmls = sorted(glob.glob(os.path.join(gdir, 'GridSquare*.xml')) or glob.glob(os.path.join(gdir, '*.xml')))
            if not xmls:
                continue
            xml_path = xmls[-1]
            info = parse_gridsquare_xml(xml_path)
            sx, sy = info.get('stage_x'), info.get('stage_y')
            if sx is None or sy is None:
                continue

            ax, ay = stage_to_atlas(sx, sy)
            m = re.search(r'GridSquare[_\s-]*([0-9]+)', os.path.basename(gdir), flags=re.IGNORECASE)
            grid_id = int(m.group(1)) if m else None

            rot = shear = atlas_w = atlas_h = phys_w = phys_h = atlas_cx = atlas_cy = None
            if (grid_id is not None) and (grid_id in nodes):
                node = nodes[grid_id]
                rot = node.get('rotation', None)
                shear = node.get('shear', None)
                atlas_w = node.get('size_w_px', None)
                atlas_h = node.get('size_h_px', None)
                phys_w = node.get('phys_w_m', None)
                phys_h = node.get('phys_h_m', None)
                atlas_cx = node.get('atlas_x', None)
                atlas_cy = node.get('atlas_y', None)

            row = {
                'grid_id': grid_id, 'folder': gdir, 'xml_path': xml_path,
                'stage_x_m': sx, 'stage_y_m': sy, 'atlas_x_px': ax, 'atlas_y_px': ay,
                'rotation_rad': rot, 'shear': shear,
                'atlas_box_w_px': atlas_w, 'atlas_box_h_px': atlas_h,
                'phys_box_w_m': phys_w, 'phys_box_h_m': phys_h,
            }
            if check_node_center and atlas_cx is not None and atlas_cy is not None:
                row['atlas_x_center_px'] = atlas_cx
                row['atlas_y_center_px'] = atlas_cy
                row['atlas_center_delta_px'] = float(np.hypot(ax - atlas_cx, ay - atlas_cy))
            rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty and 'rotation_rad' in df.columns:
        df['rotation_rad'] = pd.to_numeric(df['rotation_rad'], errors='coerce')
        if fill_rotation == 'median' and df['rotation_rad'].notna().any():
            try:
                fallback = float(df['rotation_rad'].dropna().median())
            except Exception:
                fallback = 0.0
            df['rotation_rad'] = df['rotation_rad'].fillna(fallback)
        else:
            df['rotation_rad'] = df['rotation_rad'].fillna(0.0)
        df['rotation_deg'] = np.degrees(df['rotation_rad'].values.astype(float))

    if 'shear' in df.columns:
        df['shear'] = pd.to_numeric(df['shear'], errors='coerce').fillna(0.0)

    return df, M, used_sample, used_dm_path

## Annotate_gridsquare

NS = {
    "fei": "http://schemas.datacontract.org/2004/07/Fei.SharedObjects",
    "arr": "http://schemas.microsoft.com/2003/10/Serialization/Arrays",
    "types": "http://schemas.datacontract.org/2004/07/Fei.Types",
    "common": "http://schemas.datacontract.org/2004/07/Fei.Common.Types",
    "draw": "http://schemas.datacontract.org/2004/07/System.Drawing",
    "media": "http://schemas.datacontract.org/2004/07/System.Windows.Media",
    "epu": "http://schemas.datacontract.org/2004/07/Applications.Epu.Persistence",
}

GS_IMG_RE = re.compile(r"^GridSquare_(\d{8}_\d{6})\.jpg$", re.IGNORECASE)
GS_XML_RE = re.compile(r"^GridSquare_(\d{8}_\d{6})\.xml$", re.IGNORECASE)
GS_SUPPORT_IMG_RE = re.compile(r"^GridSquare_Support_(\d{8}_\d{6})\.jpg$", re.IGNORECASE)
GS_SUPPORT_XML_RE = re.compile(r"^GridSquare_Support_(\d{8}_\d{6})\.xml$", re.IGNORECASE)
FH_XML_RE = re.compile(r"^FoilHole_(.+)_(\d{8}_\d{6})\.xml$", re.IGNORECASE)

def _ln(tag):
    return tag.split("}")[-1] if isinstance(tag, str) else tag

def _to_float(val):
    try:
        return float(val)
    except Exception:
        return None

def _safe_dirnameN(path, n):
    d = os.path.abspath(path)
    for _ in range(n):
        d = os.path.dirname(d)
    return d

def parse_timestamp(ts_str):
    return datetime.strptime(ts_str, "%Y%m%d_%H%M%S")

## From annotate_gridsquare

def get_text_float(elem):
    if elem is None or elem.text is None:
        return None
    try:
        return float(elem.text)
    except Exception:
        return None

def parse_readout_area(root):
    width = height = None
    for elem in root.iter():
        if _ln(elem.tag) == "ReadoutArea":
            for ch in elem:
                lname = _ln(ch.tag)
                if lname == "width":
                    try:
                        width = int(ch.text)
                    except Exception:
                        pass
                elif lname == "height":
                    try:
                        height = int(ch.text)
                    except Exception:
                        pass
            if width is not None and height is not None:
                return width, height
    return width, height

def parse_pixelsize(root):
    px_x = get_text_float(root.find(".//fei:SpatialScale/fei:pixelSize/fei:x/fei:numericValue", NS))
    px_y = get_text_float(root.find(".//fei:SpatialScale/fei:pixelSize/fei:y/fei:numericValue", NS))
    return px_x, px_y

def parse_ref_matrix(root):
    m11 = get_text_float(root.find(".//fei:ReferenceTransformation/fei:matrix/media:_m11", NS))
    m12 = get_text_float(root.find(".//fei:ReferenceTransformation/fei:matrix/media:_m12", NS))
    m21 = get_text_float(root.find(".//fei:ReferenceTransformation/fei:matrix/media:_m21", NS))
    m22 = get_text_float(root.find(".//fei:ReferenceTransformation/fei:matrix/media:_m22", NS))
    if None in (m11, m12, m21, m22):
        return None
    return np.array([[m11, m12], [m21, m22]], dtype=float)

def parse_stage_xy(root, override=None):
    if override and ("stage_x" in override) and ("stage_y" in override):
        return override["stage_x"], override["stage_y"]
    sx = sy = None
    sx = get_text_float(root.find(".//fei:microscopeData/fei:stage/fei:Position/fei:X", NS))
    sy = get_text_float(root.find(".//fei:microscopeData/fei:stage/fei:Position/fei:Y", NS))
    if sx is not None and sy is not None:
        return sx, sy
    for e in root.iter():
        if _ln(e.tag) == "Position":
            for cc in e:
                l = _ln(cc.tag)
                if l == "X" and sx is None:
                    sx = _to_float(cc.text)
                elif l == "Y" and sy is None:
                    sy = _to_float(cc.text)
            if sx is not None and sy is not None:
                break
    return sx, sy

def parse_imageshift(root):
    x = get_text_float(root.find(".//fei:microscopeData/fei:optics/fei:ImageShift/types:_x", NS))
    y = get_text_float(root.find(".//fei:microscopeData/fei:optics/fei:ImageShift/types:_y", NS))
    return x, y

def find_first_float_by_local_name(root, local_name):
    for elem in root.iter():
        if _ln(elem.tag) == local_name:
            try:
                return float(elem.text)
            except Exception:
                continue
    return None

# -------- GridSquare metadata --------

def parse_gridsquare_meta(xml_path):
    root = ET.parse(xml_path).getroot()
    stage_x, stage_y = parse_stage_xy(root)
    px_x, px_y = parse_pixelsize(root)
    w, h = parse_readout_area(root)
    hole_size_m_xml = find_first_float_by_local_name(root, "HoleSize")
    refM = parse_ref_matrix(root)
    imgshift_x, imgshift_y = parse_imageshift(root)
    meta = {
        "stage_x": stage_x,
        "stage_y": stage_y,
        "px_x": px_x,
        "px_y": px_y,
        "width": w,
        "height": h,
        "hole_size_m_xml": hole_size_m_xml,
        "refM": refM,
        "imageshift": (imgshift_x, imgshift_y),
    }
    return meta

def canonicalize_uniq(u):
    if u is None:
        return None
    m = re.match(r"^(\d+)$", str(u))
    return m.group(1) if m else str(u)

def parse_dm_pixelcenters_by_uniq(gs_xml_path):
    dm_path = find_gridsquare_dm_path(gs_xml_path)
    if dm_path is None or not os.path.isfile(dm_path):
        return {}

    try:
        root = ET.parse(dm_path).getroot()
    except Exception:
        return {}

    result = {}
    for node in root.iter():
        pc = None
        pwh = None
        for ch in list(node):
            ln = _ln(ch.tag)
            if ln == "PixelCenter":
                pc = ch
            elif ln == "PixelWidthHeight":
                pwh = ch

        if pc is None:
            continue

        x = y = None
        for cc in list(pc):
            ln = _ln(cc.tag).lower()
            if ln == "x":
                x = _to_float(cc.text)
            elif ln == "y":
                y = _to_float(cc.text)
        if x is None or y is None:
            continue

        w = h = None
        if pwh is not None:
            for cc in list(pwh):
                ln = _ln(cc.tag).lower()
                if ln == "width":
                    w = _to_float(cc.text)
                elif ln == "height":
                    h = _to_float(cc.text)

        uniqs_here = set()
        for sub in node.iter():
            if _ln(sub.tag) == "BaseFileName" and sub.text:
                m = re.search(r"FoilHole_(\d+)", sub.text)
                if m:
                    uniqs_here.add(canonicalize_uniq(m.group(1)))

        for uq in uniqs_here:
            if uq not in result:
                result[uq] = {"x": x, "y": y, "width": w, "height": h}

    return result

def find_latest_gridsquare_xml_relaxed(gs_dir):
    xmls = []
    for fname in os.listdir(gs_dir):
        m = GS_XML_RE.match(fname)
        ts = None
        if m:
            try:
                ts = parse_timestamp(m.group(1))
            except Exception:
                ts = None
        full = os.path.join(gs_dir, fname)
        if fname.lower().endswith(".xml"):
            if ts is not None:
                xmls.append((ts, full))
            else:
                try:
                    xmls.append((datetime.fromtimestamp(os.path.getmtime(full)), full))
                except Exception:
                    pass
    if not xmls:
        return None
    xmls.sort(key=lambda x: x[0], reverse=True)
    return xmls[0][1]

def find_gridsquare_dm_path(gs_xml_path):
    gs_id = extract_gridsquare_number_from_path(gs_xml_path)
    if not gs_id:
        return None
    session_root = _safe_dirnameN(gs_xml_path, 3)
    dm_path = os.path.join(session_root, "Metadata", f"GridSquare_{gs_id}.dm")
    return dm_path if os.path.isfile(dm_path) else None

def extract_gridsquare_number_from_path(gs_xml_path):
    gs_dir = os.path.basename(os.path.dirname(gs_xml_path))
    m = re.match(r"GridSquare_(\d+)", gs_dir, flags=re.IGNORECASE)
    return m.group(1) if m else None

## Annotate_foilhole

FOILHOLE_RE = re.compile(r"^FoilHole_([A-Za-z0-9]+)_(\d{8})_(\d{6})\.jpg$", re.IGNORECASE)
MICROGRAPH_RE = re.compile(
    r"^FoilHole_([A-Za-z0-9]+)_Data_[^_]+_[^_]+_(\d{8})_(\d{6})\.jpg$",
    re.IGNORECASE,
)

def _parse_datetime_tokens(date_str: str, time_str: str):
    try:
        return datetime.strptime(date_str + time_str, "%Y%m%d%H%M%S")
    except Exception:
        return (date_str, time_str)

def _find_gridsquares(base_folder: str):
    gs_root = os.path.join(base_folder, "Images-Disc1")
    if not os.path.isdir(gs_root):
        return []
    gs_dirs = [
        os.path.join(gs_root, d)
        for d in os.listdir(gs_root)
        if d.startswith("GridSquare") and os.path.isdir(os.path.join(gs_root, d))
    ]
    gs_dirs.sort()
    return gs_dirs

def _latest_foilhole_with_micrograph(gs_dir: str) -> Optional[Tuple[str, str]]:
    """
    Return (foilhole_jpg_path, micrograph_jpg_path) for the latest FoilHole
    in this GridSquare that has a matching micrograph, or None if none found.
    """
    holes_dir = os.path.join(gs_dir, "FoilHoles")
    data_dir = os.path.join(gs_dir, "Data")
    if not (os.path.isdir(holes_dir) and os.path.isdir(data_dir)):
        return None

    # Collect latest FoilHole JPG per key
    groups = {}
    for name in os.listdir(holes_dir):
        m = FOILHOLE_RE.match(name)
        if not m:
            continue
        key, date_str, time_str = m.group(1), m.group(2), m.group(3)
        path = os.path.join(holes_dir, name)
        dt = _parse_datetime_tokens(date_str, time_str)
        prev = groups.get(key)
        if prev is None or dt > prev[0]:
            groups[key] = (dt, path, date_str, time_str)

    if not groups:
        return None

    # For each key, see if there is a matching micrograph; keep latest by time
    candidates = []
    for key, (dt, fh_path, date_str, time_str) in groups.items():
        # Find matching micrograph in Data
        best_micro = None
        best_dt = None
        for name in os.listdir(data_dir):
            m = MICROGRAPH_RE.match(name)
            if not m:
                continue
            key_m, d_str, t_str = m.group(1), m.group(2), m.group(3)
            if key_m != key:
                continue
            dt_m = _parse_datetime_tokens(d_str, t_str)
            if best_micro is None or dt_m > best_dt:
                best_micro = os.path.join(data_dir, name)
                best_dt = dt_m
        if best_micro is not None:
            candidates.append((dt, fh_path, best_micro))

    if not candidates:
        return None

    # Latest by FoilHole time
    candidates.sort(key=lambda tup: tup[0], reverse=True)
    _, fh_path, micro_path = candidates[0]
    return fh_path, micro_path

def _find_matching_foilhole_xml(fh_jpg_path: str) -> Optional[str]:
    """
    Given a FoilHole_XXX_YYYYMMDD_HHMMSS.jpg, find the corresponding XML
    in the same FoilHoles directory (matching uniq and timestamp).
    """
    fname = os.path.basename(fh_jpg_path)
    m = FOILHOLE_RE.match(fname)
    if not m:
        return None
    key, date_str, time_str = m.group(1), m.group(2), m.group(3)
    ts_str = f"{date_str}_{time_str}"
    holes_dir = os.path.dirname(fh_jpg_path)
    xml_name = f"FoilHole_{key}_{ts_str}.xml"
    xml_path = os.path.join(holes_dir, xml_name)
    if os.path.isfile(xml_path):
        return xml_path

    # Fallback: any FoilHole_<key>_*.xml with closest timestamp
    xml_candidates = []
    for name in os.listdir(holes_dir):
        if not name.lower().endswith(".xml"):
            continue
        if not name.lower().startswith(f"foilhole_{key.lower()}_"):
            continue
        full = os.path.join(holes_dir, name)
        try:
            # Expect FoilHole_<key>_YYYYMMDD_HHMMSS.xml
            parts = name.rsplit("_", 2)
            if len(parts) < 3:
                continue
            dt_str = parts[-2] + "_" + parts[-1].split(".")[0]
            dt = datetime.strptime(dt_str, "%Y%m%d_%H%M%S")
        except Exception:
            continue
        xml_candidates.append((dt, full))

    if not xml_candidates:
        return None

    xml_candidates.sort(key=lambda t: t[0], reverse=True)
    return xml_candidates[0][1]

def _parse_foilhole_center_from_xml(xml_path: str):
    """
    Parse FoilHole XML to get the hole center in image pixels.
    Prefer a dedicated center if present; otherwise fall back to image center.
    """
    try:
        root = ET.parse(xml_path).getroot()
    except Exception:
        return None, None, None, None, None, None

    # Readout area (image size)
    w, h = parse_readout_area(root)
    px_x, px_y = parse_pixelsize(root)

    # Try to find a center element (various schemas)
    cx = cy = None
    # 1) Look for something like FindFoilHoleCenterResults / Center / x,y
    for elem in root.iter():
        if _ln(elem.tag).lower() == "findfoilholecenterresults":
            for ch in elem.iter():
                ln = _ln(ch.tag).lower()
                if ln == "center":
                    # children x,y
                    for cc in ch:
                        l2 = _ln(cc.tag).lower()
                        if l2 == "x":
                            cx = _to_float(cc.text)
                        elif l2 == "y":
                            cy = _to_float(cc.text)
                    break

    # 2) Fallback: any element named Center with x,y children
    if cx is None or cy is None:
        for elem in root.iter():
            if _ln(elem.tag).lower() == "center":
                tmp_x = tmp_y = None
                for cc in elem:
                    l2 = _ln(cc.tag).lower()
                    if l2 == "x":
                        tmp_x = _to_float(cc.text)
                    elif l2 == "y":
                        tmp_y = _to_float(cc.text)
                if tmp_x is not None and tmp_y is not None:
                    cx, cy = tmp_x, tmp_y
                    break

    # 3) If still missing, use image center if we know width/height
    if (cx is None or cy is None) and (w is not None and h is not None):
        cx = w / 2.0
        cy = h / 2.0

    return cx, cy, w, h, px_x, px_y

def _parse_template_areas_from_dm(session_dir: str):
    """
    Parse EpuSession.dm and return:
      - template_px_size: (px_width_m, px_height_m) or (None, None)
      - autofocus_shift: (dx_px, dy_px) or None
      - acquisition_shifts: list of (dx_px, dy_px)
      - drift_shift: (dx_px, dy_px) or None
    """
    dm_path = os.path.join(session_dir, "EpuSession.dm")
    if not os.path.isfile(dm_path):
        return (None, None), None, [], None

    try:
        root = ET.parse(dm_path).getroot()
    except Exception:
        return (None, None), None, [], None

    def find_first(elem, local_name):
        for e in elem.iter():
            if _ln(e.tag) == local_name:
                return e
        return None

    def parse_shift(node):
        if node is None:
            return None
        dx = dy = None
        for ch in node.iter():
            ln = _ln(ch.tag).lower()
            if ln == "width":
                dx = _to_float(ch.text)
            elif ln == "height":
                dy = _to_float(ch.text)
        if dx is None or dy is None:
            return None
        return (dx, dy)

    # TemplateImagePixelSize
    px_w = px_h = None
    tip = find_first(root, "TemplateImagePixelSize")
    if tip is not None:
        for ch in tip.iter():
            ln = _ln(ch.tag).lower()
            if ln == "width":
                px_w = _to_float(ch.text)
            elif ln == "height":
                px_h = _to_float(ch.text)

    # AutoFocusArea
    af_node = find_first(root, "AutoFocusArea")
    af_shift = None
    if af_node is not None:
        sh = find_first(af_node, "ShiftInPixels")
        af_shift = parse_shift(sh)

    # DataAcquisitionAreas
    acq_shifts: List[Tuple[float, float]] = []
    daa_node = find_first(root, "DataAcquisitionAreas")
    if daa_node is not None:
        for kv in daa_node.iter():
            ln = _ln(kv.tag)
            if "KeyValuePair" not in ln:
                continue
            value_elem = None
            for ch in kv:
                if _ln(ch.tag) == "value":
                    value_elem = ch
                    break
            if value_elem is None:
                continue
            sh = find_first(value_elem, "ShiftInPixels")
            s = parse_shift(sh)
            if s is not None:
                acq_shifts.append(s)

    # DriftStabilizationArea
    drift_node = find_first(root, "DriftStabilizationArea")
    drift_shift = None
    if drift_node is not None:
        sh = find_first(drift_node, "ShiftInPixels")
        drift_shift = parse_shift(sh)

    return (px_w, px_h), af_shift, acq_shifts, drift_shift

def _parse_micrograph_settings_from_dm(session_dir: str):
    """
    Parse EpuSession.dm to get micrograph acquisition settings for the main
    DataAcquisition / DriftMeasurement mode.

    Returns a dict with:
      {
        "beam_diameter_m": float or None,
        "cam_width_px": int or None,
        "cam_height_px": int or None,
        "binning_x": int or None,
        "binning_y": int or None,
      }
    """
    dm_path = os.path.join(session_dir, "EpuSession.dm")
    out = {
        "beam_diameter_m": None,
        "cam_width_px": None,
        "cam_height_px": None,
        "binning_x": None,
        "binning_y": None,
    }
    if not os.path.isfile(dm_path):
        return out

    try:
        root = ET.parse(dm_path).getroot()
    except Exception:
        return out

    def localname(tag):
        return tag.split("}")[-1] if "}" in tag else tag

    # Find MicroscopeSettings
    ms_root = None
    for elem in root.iter():
        if localname(elem.tag) == "MicroscopeSettings":
            ms_root = elem
            break
    if ms_root is None:
        return out

    # Iterate KeyValuePairOfExperimentSettingsIdMicroscopeSettings...
    for kv in ms_root.iter():
        ln = localname(kv.tag)
        if "KeyValuePairOfExperimentSettingsIdMicroscopeSettings" not in ln:
            continue
        key_elem = None
        val_elem = None
        for ch in kv:
            lnc = localname(ch.tag)
            if lnc == "key":
                key_elem = ch
            elif lnc == "value":
                val_elem = ch
        if key_elem is None or val_elem is None:
            continue
        key_text = (key_elem.text or "").strip()

        # Heuristic: pick DataAcquisition or DriftMeasurement as the main micrograph mode
        if key_text not in ("DataAcquisition", "DriftMeasurement", "Drift Measurement"):
            continue

        acq = None
        optics = None
        for ch in val_elem:
            lnc = localname(ch.tag)
            if lnc == "Acquisition":
                acq = ch
            elif lnc == "Optics":
                optics = ch

        # Beam diameter
        if optics is not None:
            for ch in optics.iter():
                if localname(ch.tag) == "BeamDiameter":
                    try:
                        out["beam_diameter_m"] = float(ch.text)
                    except Exception:
                        pass

        # Camera readout + binning
        if acq is not None:
            cam = None
            for ch in acq:
                if localname(ch.tag) == "camera":
                    cam = ch
                    break
            if cam is not None:
                # Binning
                for ch in cam.iter():
                    if localname(ch.tag) == "Binning":
                        bx = by = None
                        for cc in ch:
                            ln2 = localname(cc.tag)
                            if ln2 == "x":
                                try:
                                    bx = int(cc.text)
                                except Exception:
                                    pass
                            elif ln2 == "y":
                                try:
                                    by = int(cc.text)
                                except Exception:
                                    pass
                        out["binning_x"] = bx
                        out["binning_y"] = by
                    if localname(ch.tag) == "ReadoutArea":
                        w = h = None
                        for cc in ch:
                            ln2 = localname(cc.tag)
                            if ln2 == "width":
                                try:
                                    w = int(cc.text)
                                except Exception:
                                    pass
                            elif ln2 == "height":
                                try:
                                    h = int(cc.text)
                                except Exception:
                                    pass
                        out["cam_width_px"] = w
                        out["cam_height_px"] = h

        break  # stop after first matching mode

    return out

def _find_matching_micrograph_xml(micro_jpg_path: str) -> Optional[str]:
    """
    Given a FoilHole_XXX_Data_..._YYYYMMDD_HHMMSS.jpg micrograph,
    find the corresponding XML in the same directory (matching timestamp).
    """
    fname = os.path.basename(micro_jpg_path)
    m = MICROGRAPH_RE.match(fname)
    if not m:
        return None
    key, date_str, time_str = m.group(1), m.group(2), m.group(3)
    data_dir = os.path.dirname(micro_jpg_path)

    candidates = []
    for name in os.listdir(data_dir):
        if not name.lower().endswith(".xml"):
            continue
        if not name.lower().startswith(f"foilhole_{key.lower()}_"):
            continue
        full = os.path.join(data_dir, name)
        parts = name.rsplit("_", 2)
        if len(parts) < 3:
            continue
        dt_str = parts[-2] + "_" + parts[-1].split(".")[0]
        try:
            dt = datetime.strptime(dt_str, "%Y%m%d_%H%M%S")
        except Exception:
            continue
        candidates.append((dt, full))

    if not candidates:
        return None

    candidates.sort(key=lambda t: t[0], reverse=True)
    return candidates[0][1]

def _parse_micrograph_meta(xml_path: str):
    """
    Parse micrograph XML to get readout area and pixel size.
    Returns (w_px, h_px, px_x_m, px_y_m) or (None, None, None, None) on failure.
    """
    try:
        root = ET.parse(xml_path).getroot()
    except Exception:
        return None, None, None, None

    w, h = parse_readout_area(root)
    px_x, px_y = parse_pixelsize(root)
    return w, h, px_x, px_y

### Helpers specific to this notebook

Run this cell, shouldn't have to change anything

In [ ]:
FRACTIONS_RE = re.compile(
    r"^FoilHole_(?P<uniq>[A-Za-z0-9]+)_Data_(?P<t1>[^_]+)_(?P<t2>[^_]+)_(?P<date>\d{8})_(?P<time>\d{6})_Fractions\.(?P<ext>mrc|tif|tiff)$",
    re.IGNORECASE
)

def gridsquare_id_from_dir(gs_dir: str):
    m = re.search(r"GridSquare_(\d+)", os.path.basename(gs_dir), flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

def find_latest_gridsquare_xml(gs_dir: str):
    # reuse your relaxed finder
    return find_latest_gridsquare_xml_relaxed(gs_dir)

def compute_M_local(theta_rad: float, shear: float, w_px: float, h_px: float):
    """
    Match annotate_atlas_pair() logic for local transform (rotation + optional shear).
    """
    SHEAR_THRESHOLD = 0.02
    ASPECT_TOL = 0.15

    # mimic the "out_of_tol" handling
    aspect_out = False
    if w_px > 0 and h_px > 0:
        ratio = w_px / h_px
        aspect_out = (abs(ratio - 1.0) > ASPECT_TOL)
    out_of_tol = (abs(shear) > SHEAR_THRESHOLD) or aspect_out

    if out_of_tol:
        shear_val = 0.0
        side = (w_px + h_px) / 2.0
        w_px = side
        h_px = side
    else:
        shear_val = float(np.clip(shear, -SHEAR_THRESHOLD, SHEAR_THRESHOLD))

    ct = np.cos(theta_rad)
    st = np.sin(theta_rad)
    R = np.array([[ct, -st], [st, ct]], dtype=float)

    if shear_val != 0.0:
        S = np.array([[1.0, shear_val], [0.0, 1.0]], dtype=float)
        M_local = S.T @ R.T
    else:
        M_local = R.T

    return M_local, w_px, h_px

def scan_fractions_micrographs(session_dir: str):
    rows = []
    for disc in sorted(glob.glob(os.path.join(session_dir, "Images-Disc*"))):
        data_dirs = glob.glob(os.path.join(disc, "GridSquare_*", "Data"))
        for data_dir in sorted(data_dirs):
            gs_dir = os.path.dirname(data_dir)
            gs_id = gridsquare_id_from_dir(gs_dir)

            for fn in os.listdir(data_dir):
                m = FRACTIONS_RE.match(fn)
                if not m:
                    continue
                dt = datetime.strptime(m.group("date")+m.group("time"), "%Y%m%d%H%M%S")
                rows.append({
                    "gs_dir": gs_dir,
                    "gs_id": gs_id,
                    "uniq": canonicalize_uniq(m.group("uniq")),
                    "dt": dt,
                    "fractions_path": os.path.join(data_dir, fn),
                })
    return rows

def fractions_to_micro_xml(p):
    # FoilHole_*_Fractions.mrc|tif|tiff  ->  FoilHole_*.xml
    # (your examples show both exist; we prefer the non-_Fractions.xml)
    base = re.sub(r"_Fractions\.(mrc|tif|tiff)$", "", p, flags=re.IGNORECASE)
    xml = base + ".xml"
    return xml

def norm_xml(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return None
    p = str(p).strip()
    # unify separators + make absolute
    p = os.path.abspath(os.path.expanduser(p))
    p = os.path.normpath(p)
    # if df_all sometimes stores _Fractions.xml, map it to the base xml
    p = re.sub(r"_Fractions\.xml$", ".xml", p, flags=re.IGNORECASE)
    return p

def file_to_xml_candidates(file_value: str) -> list[str]:
    s = Path(str(file_value)).name

    if "_Fractions_patch_aligned_doseweighted.mrc" in s:
        s = s[: s.find("_Fractions_patch_aligned_doseweighted.mrc")]

    # strip one extension
    for ext in (".mrc", ".tif", ".tiff", ".eer", ".png", ".jpg", ".jpeg"):
        if s.lower().endswith(ext):
            s = s[: -len(ext)]
            break

    return [
        f"{s}.xml",
        f"{s}_Fractions.xml",  # sometimes you might have only this
        f"{s}.eer.xml",
    ]

def file_to_xml_path(file_value: str):
    for cand in file_to_xml_candidates(file_value):
        p = xml_index.get(cand)
        if p is not None:
            return str(p)
    return pd.NA


### Load acquisition-area offsets from EpuSession.dm, map gridsquares to atlas, and compute global coordinates for each micrograph

Run these cells, shouldn't have to change anything

In [ ]:
(template_px_size, _af_shift, acq_shifts, _drift_shift) = _parse_template_areas_from_dm(SESSION_DIR)
tpl_px_w_m, tpl_px_h_m = template_px_size  # meters per template pixel (width/height)

print("Template pixel size (m):", template_px_size)
print("Number of acquisition areas:", len(acq_shifts))

In [ ]:
df_gs, M_stage2atlas, used_sample, used_dm_path = map_grids_to_atlas(ATLAS_ROOT, SESSION_DIR)

# choose center coordinates when available (matches annotate_atlas_pair)
df_gs["ax"] = df_gs["atlas_x_center_px"].fillna(df_gs["atlas_x_px"])
df_gs["ay"] = df_gs["atlas_y_center_px"].fillna(df_gs["atlas_y_px"])

gs_lookup = df_gs.set_index("grid_id").to_dict(orient="index")
print("GridSquares mapped:", len(gs_lookup))
print("Atlas.dm used:", used_dm_path)

In [ ]:
micro_rows = scan_fractions_micrographs(SESSION_DIR)
df_micro = pd.DataFrame(micro_rows)

# one row per Fractions micrograph, shot_index determined by earliest->latest per hole
df_micro = df_micro.sort_values(["gs_id", "uniq", "dt"]).reset_index(drop=True)
df_micro["shot_index"] = df_micro.groupby(["gs_id", "uniq"]).cumcount()
df_micro.head()

# cache per GridSquare: gs_meta, dm_centers
cache = {}

out = []
for r in df_micro.to_dict(orient="records"):
    gs_id = r["gs_id"]
    uniq = canonicalize_uniq(r["uniq"])
    if gs_id not in gs_lookup:
        continue

    # load GS meta + DM pixel centers (cached)
    if gs_id not in cache:
        gs_xml = find_latest_gridsquare_xml(r["gs_dir"])
        if not gs_xml:
            cache[gs_id] = None
        else:
            gs_meta = parse_gridsquare_meta(gs_xml)
            dm_centers = parse_dm_pixelcenters_by_uniq(gs_xml)  # x,y,width,height in GS acquisition pixels
            cache[gs_id] = (gs_xml, gs_meta, dm_centers)

    if cache[gs_id] is None:
        continue

    gs_xml, gs_meta, dm_centers = cache[gs_id]
    cinfo = dm_centers.get(uniq)
    if not cinfo:
        continue

    cx = cinfo.get("x")
    cy = cinfo.get("y")
    if cx is None or cy is None:
        continue

    # optional: apply acquisition-area offset if multi-shot
    dx_m = dy_m = 0.0
    acq_idx = int(r["shot_index"]) if int(r["shot_index"]) < len(acq_shifts) else None
    if acq_idx is not None and tpl_px_w_m and tpl_px_h_m:
        dx_px_tpl, dy_px_tpl = acq_shifts[acq_idx]
        dx_m = float(dx_px_tpl) * float(tpl_px_w_m)
        dy_m = float(dy_px_tpl) * float(tpl_px_h_m)

    # convert meters -> GS acquisition pixels using GS pixel size (m/pixel)
    gs_px_x = gs_meta.get("px_x")
    gs_px_y = gs_meta.get("px_y") or gs_px_x
    if gs_px_x and gs_px_y and gs_px_x > 0 and gs_px_y > 0:
        cx = float(cx) + dx_m / float(gs_px_x)
        cy = float(cy) + dy_m / float(gs_px_y)

    # normalize within GS readout area -> local atlas-box coordinates
    W = float(gs_meta["width"])
    H = float(gs_meta["height"])
    dx_local = (cx / W - 0.5)
    dy_local = (cy / H - 0.5)

    g = gs_lookup[gs_id]
    ax = float(g["ax"])
    ay = float(g["ay"])
    theta = float(g.get("rotation_rad") or 0.0)
    shear = float(g.get("shear") or 0.0)
    w_px = float(g.get("atlas_box_w_px") or 0.0)
    h_px = float(g.get("atlas_box_h_px") or 0.0)
    if w_px <= 0 or h_px <= 0:
        continue

    M_local, w_eff, h_eff = compute_M_local(theta, shear, w_px, h_px)

    v = np.array([dx_local * w_eff, dy_local * h_eff], dtype=float)
    xy = v @ M_local + np.array([ax, ay], dtype=float)
    atlas_x, atlas_y = float(xy[0]), float(xy[1])

    # "normal graph" coords: origin at atlas center, y increases upward
    x0 = atlas_x - ATLAS_FULL_RES_PX / 2.0
    y0 = (ATLAS_FULL_RES_PX / 2.0) - atlas_y

    out.append({
        "grid_square_id": gs_id,
        "foilhole_uniq": uniq,
        "fractions_path": r["fractions_path"],
        "dt": r["dt"], 
        "shot_index": int(r["shot_index"]),
        "acq_index_0based": acq_idx,
        "atlas_x": atlas_x,
        "atlas_y": atlas_y,
        "x0_centered": x0,
        "y0_centered_up": y0,
    })

df = pd.DataFrame(out).sort_values(["dt", "grid_square_id", "foilhole_uniq"])
df["date"] = df["dt"].dt.strftime("%Y%m%d")
df["time"] = df["dt"].dt.strftime("%H%M%S")
df["xml_path"] = df["fractions_path"].map(fractions_to_micro_xml)
df["xml_key"] = df["xml_path"].map(norm_xml)
#df.to_csv(OUT_CSV, index=False)

df_positions = df

In [ ]:
df_positions.head(2)

In [ ]:
x_axis = 'x0_centered'
y_axis = 'y0_centered_up'

plt.scatter(df[x_axis], df[y_axis], alpha=1, color='blue', s=1)

plt.title('Micrograph Locations (Atlas Level)')
plt.xlabel('Relative Position X')
plt.ylabel('Relative Position Y')
plt.grid(False)
plt.xlim(-2000,2000)
plt.ylim(-2000,2000)
#plt.savefig(f"{OUTDIR}/Square_locations.png")   # <-- unhash if you want to save image
plt.show()

# PART II: Parse CTF fits from CryoSPARC

Run these cells, shouldn't have to change anything

In [ ]:
df_all = pd.read_csv(csv_path)

if "Index" in df_all.columns:
    df_all["Index"] = pd.to_numeric(df_all["Index"], errors="coerce")

dataset = Path(csv_path).stem

In [ ]:
session_data_dirs = list(Path(SESSION_DIR).glob("Images-Disc*/GridSquare_*/Data"))
xml_index = {}

for data_dir in session_data_dirs:
    for p in data_dir.glob("*.xml"):
        # main key
        xml_index.setdefault(p.name, p)

        # aliases: map *_Fractions.xml -> base .xml
        if p.name.lower().endswith("_fractions.xml"):
            alias = re.sub(r"_Fractions\.xml$", ".xml", p.name, flags=re.IGNORECASE)
            xml_index.setdefault(alias, p)

df_all["xml_path"] = df_all["File"].map(file_to_xml_path)
df_all["xml_key"]  = df_all["xml_path"].map(norm_xml)

In [ ]:
df_all.head(2)

In [ ]:
df_positions.head(2)

In [ ]:
df_merged = df_all.merge(
    df_positions.drop(columns=["xml_path"], errors="ignore"),
    on="xml_key",
    how="left",
)

In [ ]:
df_merged.head(2)

## Final Visualizations

You may want to change the following in these cells:
1. vmin/vmax (minimum and maximum CTF values for color scale)
2. xlim/ylim (axis bounds, particularly to zoom in on a grid square)
3. s (size of dots)

Remove the hash to save any of the figures once they look as desired.

In [ ]:
x_axis = 'x0_centered'
y_axis = 'y0_centered_up'

plt.scatter(df_merged[x_axis], df_merged[y_axis], c=df_merged["CTF Fit"], alpha=1, vmin=4, vmax=10, cmap='Blues_r', s=3)
cbar = plt.colorbar()

cbar.set_label("CTF Fit")
plt.xlabel('Relative Position X')
plt.ylabel('Relative Position Y')
plt.grid(False)
plt.xlim(-2000,2000)
plt.ylim(-2000,2000)
#plt.savefig(f"{OUTDIR}/Square_locations_CTF.png")
plt.show()

In [ ]:
x_axis = 'x0_centered'
y_axis = 'y0_centered_up'

plt.scatter(df_merged[x_axis], df_merged[y_axis], c=df_merged["CTF Fit"], alpha=1, vmin=4, vmax=10, cmap='Blues_r', s=45)
cbar = plt.colorbar()

plt.title("GS 1")
cbar.set_label("CTF Fit")
plt.xlabel('Relative Position X')
plt.ylabel('Relative Position Y')
plt.grid(False)
plt.xlim(-1225,-1075)
plt.ylim(-450,-300)
#plt.savefig(f"{OUTDIR}/Square_locations_zoom.png")
plt.show()